In [1]:
%load_ext autoreload
%autoreload 2

# IMPORTS

In [2]:
# Project setup
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import RANDOM_STATE
from src.preprocessing import (
    TextCleaner,
    create_preprocessor,
    prepare_data,
    prepare_target,
)

# Load data & Split data

In [4]:
data_path = project_root / "data" / "Airline_review.csv"
raw_df = pd.read_csv(data_path)

# remove duplicates
raw_df = raw_df.drop_duplicates(subset=raw_df.columns.drop('Unnamed: 0'))

# Split the raw data into training-validation and test sets (80/20)
train_val_df, test_df = train_test_split(
    raw_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=raw_df['Recommended']
)

# Split the training-validation data into training and validation sets (80/20)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_val_df['Recommended']
)

# Data preprocessing

## 1\. Dataset preparation

In [5]:
# remove unused features
cols_to_drop = [
    'Unnamed: 0',
    'Review Date',
    'Verified',
    'Route',
    'Airline Name',
    'Overall_Rating',
    'Aircraft',
    'Date Flown'
]

train_df = prepare_data(train_df, cols_to_drop)
val_df = prepare_data(val_df, cols_to_drop)
test_df = prepare_data(test_df, cols_to_drop)

## 2\. Target preparation

In [6]:
train_df = prepare_target(train_df)
val_df = prepare_target(val_df)
test_df = prepare_target(test_df)

## 3\. Feature analysis and preprocessing decisions

In [7]:
# target columns
target_col = 'Recommended_num'
target_cat_col = 'Recommended'

### 3\.1 Numeric features

In [8]:
num_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
            'Ground Service', 'Inflight Entertainment', 'Wifi & Connectivity',
            'Value For Money']

In [9]:
print('Missing values in Numeric columns:')
for col in num_cols:
    print(f'{col}: {train_df[col].isnull().mean() * 100:.2f}%')

Missing values in Numeric columns:
Seat Comfort: 17.65%
Cabin Staff Service: 18.09%
Food & Beverages: 37.11%
Ground Service: 20.61%
Inflight Entertainment: 53.18%
Wifi & Connectivity: 74.41%
Value For Money: 4.27%


- Missing values in service ratings are likely related to service availability rather than random data loss. Therefore, they will be imputed with a constant value (-1) during preprocessing to preserve this information.

- All rating features use the same 0–5 scale. If needed, scaling will be applied later in the preprocessing pipeline.

### 3\.2 Categorical features

In [10]:
cat_cols = ['Type Of Traveller', 'Seat Type']

In [11]:
print('Missing values in Categorical columns:')
for col in cat_cols:
    print(f'{col}: {train_df[col].isnull().mean() * 100:.2f}%')

Missing values in Categorical columns:
Type Of Traveller: 16.19%
Seat Type: 4.42%


- Missing values in categorical features will be replaced with ***Unknown***.

- The features will be encoded using `One-Hot Encoding` during preprocessing.

### 3\.3 Text features

In [12]:
# Create the text feature used in the baseline model
train_df['Review_Text'] = train_df['Review']
val_df['Review_Text'] = val_df['Review']
test_df['Review_Text'] = test_df['Review']

text_col = 'Review_Text'

At the baseline stage, the text feature is created from the review body only (`Review`). Since the preprocessing pipeline operates on a single text column, this feature can later be redefined (e.g., by combining `Review_Title` and `Review`) without modifying the pipeline itself.

## 4\. Text preprocessing

### 4\.1 Text cleaning examples

In [13]:
text_cleaner = TextCleaner()

sample = train_df['Review_Text'].sample(5, random_state=RANDOM_STATE)

cleaned = text_cleaner.fit_transform(sample)

check_df = pd.DataFrame({
    'original_text': sample,
    'cleaned_text': cleaned
})

with pd.option_context(
        'display.max_colwidth', None,
        'display.max_columns', None
):
    display(check_df)

,original_text,cleaned_text
9507,"""be using Firefly again"" A convenient flight from Singapore to Kuala Lumpur. Flight into Subang is convenient as the airport is smaller than KLIA and closer to Kuala Lumpur city. Check in was easy and service on board efficient and friendly. A small snack and a drink offered for no extra charge. This gesture is highly appreciated as this is rarely seen in short haul flights in Europe. I will be using Firefly again.","""be using Firefly again"" A convenient flight from Singapore to Kuala Lumpur. Flight into Subang is convenient as the airport is smaller than KLIA and closer to Kuala Lumpur city. Check in was easy and service on board efficient and friendly. A small snack and a drink offered for no extra charge. This gesture is highly appreciated as this is rarely seen in short haul flights in Europe. I will be using Firefly again."
7889,Santiago de Cuba-Havana in an IL96. We arrived very early at the airport as recommended and the check-in was quick and efficient but not very friendly. The flight was a domestic continuation of a flight from Paris and operated on time. The seats on board had adequate legroom but a hard bar across the lower back made them rather uncomfortable. While the boarding passes had allocated seats in fact passengers could sit anywhere except in the front area which was for the international passengers (some young men not in uniform seemed to be security staff enforcing non-mixing of international and domestic passengers). For most of the flight the cabin lights were totally extinguished (it was an evening flight) and the personal reading lights didn't work. The cabin crew disappeared while the plane was in flight. No refreshments were served. On arrival the plane docked at the international terminal where as ensured by the security staff the passengers from Paris disembarked via an air bridge while we were bussed a long way to the domestic terminal and waited a long time for our luggage. Overall Cubana is an effective and interesting way to get from A to B but don't expect any frills. By the way our booking through an international booking engine worked fine.,Santiago de Cuba-Havana in an IL96. We arrived very early at the airport as recommended and the check-in was quick and efficient but not very friendly. The flight was a domestic continuation of a flight from Paris and operated on time. The seats on board had adequate legroom but a hard bar across the lower back made them rather uncomfortable. While the boarding passes had allocated seats in fact passengers could sit anywhere except in the front area which was for the international passengers (some young men not in uniform seemed to be security staff enforcing non-mixing of international and domestic passengers). For most of the flight the cabin lights were totally extinguished (it was an evening flight) and the personal reading lights didn't work. The cabin crew disappeared while the plane was in flight. No refreshments were served. On arrival the plane docked at the international terminal where as ensured by the security staff the passengers from Paris disembarked via an air bridge while we were bussed a long way to the domestic terminal and waited a long time for our luggage. Overall Cubana is an effective and interesting way to get from A to B but don't expect any frills. By the way our booking through an international booking engine worked fine.
21902,"""customer service is very rude"" Terrible airline. I had a flights from Sydney to Denpasar and my flight was 5.40pm and i got messages from Virgin Australia customer service it says my flight is cancel and new flight is on 1pm to Brisbane and then Denpasar. Actually Sydney to Denpasar flight were not cancel, they sold my ticket to someone else for more expensive. And they took my flights much cheaper fares and flight time is earlier. Actually to Denpasar ticket via from Brisbane much cheaper then direct flight. And also they rescheduled my flight just before one

### 4\.2 Preprocessing decisions

In [14]:
# checking for the presence of URLs
url_mask = train_df['Review_Text'].str.contains(
    r'https?://|www\.',
    regex=True,
    case=False,
    na=False
)

url_mask.sum()

np.int64(6)

## **Preprocessing decisions:**

- Template review titles such as " < Airline> customer review" were removed because they do not provide useful information for classification. For the baseline model, the text feature is created from the review body only. The preprocessing pipeline operates on a single text column, allowing alternative text representations (e.g., combining review title and review text) to be evaluated in later) to be evaluated in later experiments.

- Numbers were preserved during text preprocessing because some numerical expressions may provide useful contextual information (e.g., delays, prices, aircraft models, or ratings). Their impact on model performance will be evaluated during model development.

- No HTML tags were found in the current dataset during exploration. Therefore, this preprocessing step does not affect the current data. However, HTML removal is included in the text cleaning pipeline to handle possible formatting artifacts in future data.

- A small number of URLs were found in the review texts (6 cases). Since links do not provide meaningful information for sentiment classification and may introduce unnecessary noise, they are removed during text preprocessing. This step also helps handle possible URLs in future data.

- Different apostrophe characters were found in the review texts. All variants were normalized to the standard apostrophe (') to ensure consistent text representation.

- Different types of quotation marks were also normalized to the standard double quote ("). This provides a consistent text representation while preserving the original content.

- Repeated punctuation (e.g., multiple exclamation or question marks) was reduced to a single symbol, and repeated emoticons were normalized. This removes formatting inconsistencies while preserving the presence of emphasis or emotion.

- Whitespace was normalized by replacing line breaks and tab characters with spaces, collapsing multiple consecutive spaces into a single space, and trimming leading and trailing whitespace. This ensures a clean and consistent text format without altering the semantic content.

- Standalone punctuation marks were removed to reduce noise from isolated symbols while preserving punctuation that may carry emotional information, such as exclamation marks, question marks, and emoticons.

- Text features are cleaned using the custom text preprocessing pipeline, followed by TF-IDF vectorization with unigrams and bigrams. Different text representations can be supplied to the pipeline without modifying the preprocessing steps.


# 5. Preprocessing pipeline

## Preprocessing summary

The preprocessing pipeline combines three types of features:

- Numerical features:
  - missing values are replaced with -1, as 0 is a valid rating value;
  - scaling is not applied at this stage.

- Categorical features:
  - missing values are replaced with "Unknown";
  - categories are encoded using One-Hot Encoding;
  - unknown categories are ignored during transformation.

- Text features:
  - review title and review text are combined into a single text feature;
  - text is cleaned and transformed using TF-IDF with unigrams and bigrams.

In [15]:
# define input features and target
input_cols = num_cols + cat_cols + [text_col]

X_train = train_df[input_cols].copy()
X_val = val_df[input_cols].copy()
X_test = test_df[input_cols].copy()

y_train = train_df[target_col]
y_val = val_df[target_col]
y_test = test_df[target_col]

## Preprocessing validation

In [16]:
preprocessor = create_preprocessor(
    num_cols,
    cat_cols,
    text_col
)
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_val_processed.shape, X_test_processed.shape

((14752, 152286), (3688, 152286), (4611, 152286))

The final feature matrix combines numerical, categorical and TF-IDF text features.

The fitted preprocessing pipeline was applied to validation and test sets using `transform()` only. All datasets produced the same number of features, confirming consistent feature generation.

The fitted pipeline produces a consistent feature space for training, validation, and test data.

The complete preprocessing pipeline is implemented in src/preprocessing.py and instantiated using create_preprocessor()